# A2A with Strands: Agents That Talk to Agents

You have built single agents. You have built multi-agent systems with Swarm, Graph, and Agents-as-Tools.

Every one of those lives in **one Python process**. One `pip install`. One codebase. One deploy.

A2A breaks that wall. It lets an agent call another agent that:

- runs in a **different process**
- was built by a **different team**
- uses a **different framework** (Google ADK, OpenAI Agents SDK, LangGraph)
- sits behind a **different deploy**

The only thing they share is a URL.

**One mental model for the whole notebook:**

> MCP connects an agent **down** to tools and data.
> A2A connects an agent **across** to other agents.

By the end you will stand up an A2A server, talk to it four different ways, and have your TravelMind support agent delegate a refund to a peer agent it has never imported.


## What you will build

| Part | You build | Needs AWS Bedrock? |
|---|---|---|
| A | A Refund agent exposed as an A2A server | No (boot + discovery are credential-free) |
| B | Four ways to consume it from a client | Yes (model calls) |
| C | TravelMind orchestrator delegating a refund over A2A | Yes |
| D | The decision: when A2A beats Swarm/Graph/MCP, when it does not | No |

Run Part A even with no credentials. The Agent Card and discovery work without ever calling a model.


## Setup

Two installs. The `[a2a]` extra adds the server. The `[a2a_client]` extra adds the client tools.

In [ ]:
# Run once. Skip if already installed.
%pip install -q 'strands-agents[a2a]' 'strands-agents-tools[a2a_client]'

In [ ]:
from importlib.metadata import version
for pkg in ["strands-agents", "a2a-sdk", "strands-agents-tools"]:
    print(pkg, version(pkg))

# This notebook was verified against:
#   strands-agents 1.42.0 | a2a-sdk 0.3.26 | strands-agents-tools 0.8.0

### Bedrock config (read before Part B)

The model calls in Parts B and C hit Amazon Bedrock. Two things must be true:

1. **Credentials** for AWS account `452203592848`, region `us-east-1`, with `bedrock:InvokeModel` on the model below.
2. **An inference profile ID, not a bare model ID.**

```
us.anthropic.claude-haiku-4-5-20251001-v1:0     <- works (inference profile)
anthropic.claude-haiku-4-5-20251001-v1:0        <- ValidationException
```

The `us.` prefix is the cross-region inference profile. A bare model ID throws `ValidationException` on Converse. This single character has cost people hours.

In [ ]:
import os
os.environ.setdefault("AWS_REGION", "us-east-1")

# Default model for this notebook. Swap to Sonnet 4 for harder reasoning.
MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
SERVER_URL = "http://127.0.0.1:9000"
CARD_PATH = "/.well-known/agent-card.json"
print("model:", MODEL)
print("server:", SERVER_URL)

## The four core objects (30-second version)

A2A has exactly four nouns. Learn these and the protocol stops being mysterious.

| Object | What it is | Analogy |
|---|---|---|
| **Agent Card** | JSON at `/.well-known/agent-card.json` describing who the agent is and what it can do | A business card |
| **Message** | A single turn you send to the agent | A text |
| **Task** | A unit of work the agent tracks to completion (can be long-running) | A support ticket |
| **Artifact** | A produced output attached to a task | The deliverable |

Transport is HTTP plus JSON-RPC 2.0. Discovery is the Agent Card. That is the whole wire model.


---
## Part A: Build an A2A server

We expose a **Refund agent** for the airline. It has two tools and knows nothing about who will call it.

### The one gotcha that breaks discovery

Tools become A2A **skills** on the Agent Card automatically. The card reads them straight from the agent's tool registry.

But Strands only registers a tool if it carries the `@tool` decorator. Pass a **plain function** and Strands logs `unrecognized tool specification`, registers nothing, and your card ships with `skills: []`.

We tested both. Decorated tools showed up as skills. Plain functions vanished silently. Always decorate.

In [ ]:
%%writefile refund_server.py
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer
from strands.models import BedrockModel

# @tool is mandatory. Without it the tool never registers
# and the Agent Card advertises zero skills.
@tool
def look_up_booking(pnr: str) -> dict:
    """Look up an airline booking by PNR. Returns fare and refund eligibility."""
    return {"pnr": pnr, "fare_usd": 420.0, "refundable": True, "penalty_usd": 50.0}

@tool
def process_refund(pnr: str, amount_usd: float) -> dict:
    """Process a refund for a booking. Returns a confirmation id."""
    return {"pnr": pnr, "refunded_usd": amount_usd, "confirmation": "RF-" + pnr}

agent = Agent(
    name="Refund Agent",
    description="Handles airline refund eligibility and processing.",
    model=BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    tools=[look_up_booking, process_refund],
    callback_handler=None,   # quiet: no streaming prints from the server-side agent
)

# Defaults: host=127.0.0.1, port=9000, served at root "/", streaming on.
server = A2AServer(agent=agent)

if __name__ == "__main__":
    server.serve()

`A2AServer(agent=agent)` wraps the Strands agent and exposes:

- `POST /` for JSON-RPC messages
- `GET /.well-known/agent-card.json` for discovery (current spec)
- `GET /.well-known/agent.json` for the legacy alias

Now launch it. The server is a separate process, exactly as it would run in production. We poll the card endpoint until it answers, so we never race the client against a half-booted server.

In [ ]:
import subprocess, sys, time, urllib.request

def launch_a2a_server(script: str, ready_url: str, timeout: int = 90):
    """Start an A2A server as a subprocess and block until its card endpoint is live."""
    proc = subprocess.Popen([sys.executable, script])
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"server exited early with code {proc.returncode}")
        try:
            with urllib.request.urlopen(ready_url, timeout=2) as r:
                if r.status == 200:
                    print("server ready ->", ready_url)
                    return proc
        except Exception:
            time.sleep(1)
    proc.terminate()
    raise RuntimeError("server did not become ready in time")

# sys.executable must be the same environment where strands is installed.
server_proc = launch_a2a_server("refund_server.py", SERVER_URL + CARD_PATH)

### Discovery: read the Agent Card

This is A2A's whole discovery story. One HTTP GET. No SDK, no credentials, no model call. Any client in any language can do this.

In [ ]:
import json, urllib.request

with urllib.request.urlopen(SERVER_URL + CARD_PATH) as r:
    card = json.load(r)

print("name:        ", card["name"])
print("description: ", card["description"])
print("protocol:    ", card["protocolVersion"])
print("transport:   ", card["preferredTransport"])
print("streaming:   ", card["capabilities"]["streaming"])
print("skills:      ", [s["name"] for s in card["skills"]])

# Verified live: protocol 0.3.0, transport JSONRPC, both tools present as skills.

---
## Part B: Consume it, four ways

Same server. Four client styles, from most explicit to most autonomous. Parts B and C call the model, so they need Bedrock credentials.

| Way | Class | Who decides what to call |
|---|---|---|
| B1 Direct sync | `A2AAgent` | You |
| B2 Direct streaming | `A2AAgent.stream_async` | You, with live events |
| B3 Tool provider | `A2AClientToolProvider` | The orchestrator LLM, autonomously |
| B4 Agent-as-tool | `A2AAgent` wrapped in `@tool` | The orchestrator LLM, with a clean interface |


### B1: Direct call (synchronous)

The simplest path. Construct an `A2AAgent` pointed at the URL, call it like a function, read `result.message`. The client fetches the card, sends a JSON-RPC message, and waits.

In [ ]:
from strands.agent.a2a_agent import A2AAgent

refund = A2AAgent(endpoint=SERVER_URL)

result = refund("Is PNR JX48Q2 refundable, and if so for how much after any penalty?")
print(result.message)

### B2: Streaming, and the trap inside it

`A2AAgent.stream_async` does **not** yield text deltas like a normal Strands agent. It yields raw A2A protocol events, then one final result.

The shape, confirmed from the 1.42.0 source:

```
{"type": "a2a_stream", "event": <A2A protocol object>}   # zero or more
{"result": AgentResult}                                  # always emitted LAST
```

**The trap:** older docs and tutorials read `event["data"]`. That key does not exist here. The streamed payload is under `event["event"]`, and the final answer is in the **last** event under `event["result"].message`. Branch on `event.get("type")` first, then check for `"result"`.

In [ ]:
import asyncio

async def stream_refund():
    refund = A2AAgent(endpoint=SERVER_URL)
    final = None
    async for event in refund.stream_async("Check refund eligibility for PNR JX48Q2."):
        if event.get("type") == "a2a_stream":
            # raw A2A protocol event; the payload is event["event"], NOT event["data"]
            print(".", end="", flush=True)
        elif "result" in event:                 # always the last event
            final = event["result"].message
    print()
    print(final)

await stream_refund()   # top-level await works in Jupyter; use asyncio.run(...) in a script

### B3: Let an orchestrator discover and call on its own

`A2AClientToolProvider` hands an agent three tools and gets out of the way:

- `a2a_discover_agent`: fetch a peer's card by URL
- `a2a_list_discovered_agents`: list what it has found
- `a2a_send_message`: send a message to a discovered peer

You give the orchestrator known URLs. It decides when to discover and when to call. No hard-coded peer logic.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.a2a_client import A2AClientToolProvider

provider = A2AClientToolProvider(known_agent_urls=[SERVER_URL])
print("injected tools:", [t.tool_name for t in provider.tools])

orchestrator = Agent(
    name="SupportRouter",
    description="Routes customer requests to the right specialist agent.",
    model=BedrockModel(model_id=MODEL),
    tools=provider.tools,
    callback_handler=None,
)

answer = orchestrator(
    "Discover the agent at the known URL, then ask it whether PNR JX48Q2 "
    "is refundable and for how much."
)
print(answer.message)

### B4: Wrap a peer as a single clean tool

B3 is flexible but exposes three generic tools. Often you want the peer to look like **one purpose-built tool** in your orchestrator. Wrap the `A2AAgent` call in a `@tool`.

This is the pattern Part C uses, because it reads the cleanest in a real orchestrator.

In [ ]:
from strands import tool

# name= is useful here; without it A2AAgent.name is None until a card is fetched.
refund_peer = A2AAgent(endpoint=SERVER_URL, name="refund_agent")

@tool
def ask_refund_agent(question: str) -> str:
    """Delegate any refund eligibility or processing question to the Refund agent."""
    result = refund_peer(question)     # synchronous A2A call to the peer
    return str(result.message)

print("wrapped peer as tool:", ask_refund_agent.tool_name)

---
## Part C: TravelMind delegates a refund over A2A

The anchor scenario. **TravelMind** is the airline's front-line support agent. A passenger wants to cancel.

TravelMind does **not** know refund rules. It does not import the refund code. It only knows there is a refund specialist it can reach as a tool. It delegates, gets the answer, and replies to the passenger.

That is the entire point of A2A: TravelMind and the Refund agent are two separate services that never shared a line of code.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

travelmind = Agent(
    name="TravelMind",
    description="Front-line airline customer support. Delegates specialist work to peer agents.",
    system_prompt=(
        "You are TravelMind, an airline support agent. "
        "For anything about refunds or cancellations, you MUST use the ask_refund_agent tool. "
        "Never guess refund amounts yourself. Reply to the passenger in plain, warm language."
    ),
    model=BedrockModel(model_id=MODEL),
    tools=[ask_refund_agent],     # the wrapped peer from B4
    callback_handler=None,
)

reply = travelmind(
    "Hi, I need to cancel my flight. My booking reference is JX48Q2. "
    "Will I get money back, and how much?"
)
print(reply.message)

What just happened, end to end:

```mermaid
sequenceDiagram
    participant P as Passenger
    participant T as TravelMind
    participant R as Refund Agent (A2A peer)
    P->>T: Cancel JX48Q2, how much back?
    T->>R: ask_refund_agent("refund for JX48Q2?")
    R->>R: look_up_booking + process_refund
    R-->>T: refundable, $370 after $50 penalty
    T-->>P: warm plain-language answer
```

TravelMind never knew the refund math. It knew a peer that did.


---
## Part D: When A2A, when not

A2A is a network protocol with real overhead: HTTP hops, serialization, separate deploys, auth. Do not reach for it when a function call would do.

| Your situation | Use | Why |
|---|---|---|
| Agent needs a tool or data source | **MCP** | A2A is for agents, not tools |
| Several agents, one codebase, one deploy | **Swarm / Graph / Agents-as-Tools** | In-process is faster and simpler |
| Agents owned by different teams | **A2A** | No shared imports or release train |
| Agents in different frameworks (ADK, OpenAI SDK) | **A2A** | The protocol is the contract |
| Agents must scale or deploy independently | **A2A** | Separate services, separate lifecycles |
| One latency-sensitive call, same machine | **In-process** | A network hop you do not need |

The honest rule:

> Stay in-process until an organizational or deployment wall forces you out. When it does, A2A is the door.

**Skeptic's corner:** "Is A2A just microservices with extra JSON?" Partly, yes. The value is not the transport. It is the **standard discovery and message contract**, so a Strands agent can call a Google ADK agent it has never seen, with no custom glue. If all your agents are Python in one repo, A2A buys you nothing today. The day a second team ships an agent in a second framework, it buys you everything.


## Teardown

Stop the server subprocess.

In [ ]:
server_proc.terminate()
try:
    server_proc.wait(timeout=10)
except Exception:
    server_proc.kill()
print("server stopped")

## Recap

- A2A connects agents **across** processes, teams, and frameworks. MCP connects an agent **down** to tools.
- An agent becomes an A2A server with `A2AServer(agent=agent).serve()`. Tools become skills, but **only with `@tool`**.
- Discovery is one HTTP GET to the Agent Card. No SDK, no credentials.
- Consume a peer four ways: direct sync, streaming (final answer is the **last** event under `"result"`), the tool provider, or a clean `@tool` wrapper.
- TravelMind delegated a refund to a peer it never imported. That is the whole promise.

**Next:** take this exact server, make it AgentCore-ready, and deploy it so the peer lives in AWS instead of localhost.
